# 🧱 DCGAN - Bricks Data

In this notebook, we'll walk through the steps required to train your own DCGAN on the bricks dataset

In [ ]:
%load_ext autoreload
%autoreload 2
# TensorBoard imports TensorFlow whenever it is installed, and TensorFlow and PyTorch
# in one process can crash. This makes TensorBoard use its own TF-free version instead
import sys, types
sys.modules["tensorboard.compat.notf"] = types.ModuleType("tensorboard.compat.notf")

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import lightning as L
from lightning.pytorch.callbacks import ModelCheckpoint, TQDMProgressBar
from lightning.pytorch.loggers import TensorBoardLogger

import sys
from pathlib import Path

# Make the repo root importable so `notebooks.utils_pytorch` can be found
REPO_ROOT = Path.cwd().resolve().parents[2]
sys.path.insert(0, str(REPO_ROOT))

from notebooks.utils_pytorch import display, sample_batch

# Lightning's Trainer picks the best available device
# automatically (accelerator="auto"), so we only report what it will find
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"MPS available: {torch.backends.mps.is_available()}")

## 0. Parameters <a name="parameters"></a>

In [ ]:
IMAGE_SIZE = 64
CHANNELS = 1
BATCH_SIZE = 128
Z_DIM = 100
EPOCHS = 300
LOAD_MODEL = False
ADAM_BETA_1 = 0.5
ADAM_BETA_2 = 0.999
LEARNING_RATE = 0.0002
NOISE_PARAM = 0.1

## 1. Prepare the data <a name="prepare"></a>

In [ ]:
class ImageDataset(Dataset):
    """
    All image files found under `folder` (searched recursively, in sorted order),
    resized to `image_size` - the PyTorch counterpart of Keras'
    image_dataset_from_directory. Returns uint8 tensors of shape (C, H, W).
    """

    def __init__(self, folder, image_size, color_mode="rgb", labels=None):
        exts = {".jpg", ".jpeg", ".png", ".bmp", ".gif"}
        self.paths = sorted(p for p in Path(folder).rglob("*") if p.suffix.lower() in exts)
        self.mode = {"grayscale": "L", "rgb": "RGB"}[color_mode]
        self.resize = transforms.Resize(
            image_size, interpolation=transforms.InterpolationMode.BILINEAR
        )
        self.labels = labels
        self.transform = None  # optional function applied to each image
        self.target_transform = None  # optional function applied to each label
        print(f"Found {len(self.paths)} files.")

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert(self.mode)
        img = transforms.functional.pil_to_tensor(self.resize(img))
        if self.transform:
            img = self.transform(img)
        if self.labels is None:
            return img
        label = self.labels[idx]
        if self.target_transform:
            label = self.target_transform(label)
        return img, label


train_data = ImageDataset(
    "../../../data/lego-brick-images/dataset/",
    image_size=(IMAGE_SIZE, IMAGE_SIZE),
    color_mode="grayscale",
)

In [ ]:
def preprocess(img):
    """
    Normalize the images to [-1, 1]
    """
    img = (img.float() - 127.5) / 127.5
    return img


# Like Keras' .map(): preprocess is applied to every image as it is loaded.
# The DataLoader groups the images into shuffled batches.
train_data.transform = preprocess
train = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
train_sample = sample_batch(train)

In [ ]:
display(train_sample)

## 2. Build the GAN <a name="build"></a>

In [ ]:
# PyTorch images are channels-first: (batch, channels, height, width).
# padding=1 with kernel 4 / stride 2 halves the size, like Keras' padding="same".
discriminator = nn.Sequential(
    nn.Conv2d(CHANNELS, 64, kernel_size=4, stride=2, padding=1, bias=False),
    nn.LeakyReLU(0.2),
    nn.Dropout(0.3),
    nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1, bias=False),
    # PyTorch momentum = 1 - Keras momentum (Keras 0.9 -> PyTorch 0.1)
    nn.BatchNorm2d(128, momentum=0.1),
    nn.LeakyReLU(0.2),
    nn.Dropout(0.3),
    nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(256, momentum=0.1),
    nn.LeakyReLU(0.2),
    nn.Dropout(0.3),
    nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(512, momentum=0.1),
    nn.LeakyReLU(0.2),
    nn.Dropout(0.3),
    # No sigmoid here: the discriminator outputs a raw score (logit) and
    # nn.BCEWithLogitsLoss applies the sigmoid inside the loss (more stable)
    nn.Conv2d(512, 1, kernel_size=4, stride=1, padding=0, bias=False),
    nn.Flatten(),
)

print(discriminator)
print(f"Total params: {sum(p.numel() for p in discriminator.parameters()):,}")

In [ ]:
generator = nn.Sequential(
    nn.Unflatten(1, (Z_DIM, 1, 1)),  # (batch, Z_DIM) -> (batch, Z_DIM, 1, 1)
    nn.ConvTranspose2d(Z_DIM, 512, kernel_size=4, stride=1, padding=0, bias=False),
    nn.BatchNorm2d(512, momentum=0.1),
    nn.LeakyReLU(0.2),
    # padding=1 with kernel 4 / stride 2 doubles the size, like Keras' padding="same"
    nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(256, momentum=0.1),
    nn.LeakyReLU(0.2),
    nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(128, momentum=0.1),
    nn.LeakyReLU(0.2),
    nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1, bias=False),
    nn.BatchNorm2d(64, momentum=0.1),
    nn.LeakyReLU(0.2),
    nn.ConvTranspose2d(64, CHANNELS, kernel_size=4, stride=2, padding=1, bias=False),
    nn.Tanh(),
)

print(generator)
print(f"Total params: {sum(p.numel() for p in generator.parameters()):,}")

In [ ]:
class DCGAN(L.LightningModule):
    def __init__(self, discriminator, generator, latent_dim):
        super().__init__()
        self.save_hyperparameters(ignore=["discriminator", "generator"])
        self.discriminator = discriminator
        self.generator = generator
        self.latent_dim = latent_dim
        # Takes logits; unlike nn.BCELoss it also accepts the noisy labels
        # below, which fall slightly outside [0, 1]
        self.loss_fn = nn.BCEWithLogitsLoss()
        # Two networks with two optimizers: we call backward() and step()
        # ourselves instead of letting Lightning do it ("manual optimization")
        self.automatic_optimization = False

    def forward(self, z):
        return self.generator(z)

    def training_step(self, real_images, batch_idx):
        d_optimizer, g_optimizer = self.optimizers()

        # Sample random points in the latent space
        batch_size = real_images.shape[0]
        random_latent_vectors = torch.randn(batch_size, self.latent_dim, device=self.device)
        generated_images = self.generator(random_latent_vectors)

        # Train the discriminator on real and fake images.
        # .detach() stops the gradients flowing back into the generator.
        real_predictions = self.discriminator(real_images)
        fake_predictions = self.discriminator(generated_images.detach())

        real_labels = torch.ones_like(real_predictions)
        real_noisy_labels = real_labels + NOISE_PARAM * torch.rand_like(real_predictions)
        fake_labels = torch.zeros_like(fake_predictions)
        fake_noisy_labels = fake_labels - NOISE_PARAM * torch.rand_like(fake_predictions)

        d_real_loss = self.loss_fn(real_predictions, real_noisy_labels)
        d_fake_loss = self.loss_fn(fake_predictions, fake_noisy_labels)
        d_loss = (d_real_loss + d_fake_loss) / 2.0

        d_optimizer.zero_grad()
        self.manual_backward(d_loss)  # instead of d_loss.backward()
        d_optimizer.step()

        # Train the generator: it wants the discriminator to call its images real.
        # (Keras computes both losses from one forward pass; the usual PyTorch
        # pattern re-scores the fakes with the just-updated discriminator.)
        g_predictions = self.discriminator(generated_images)
        g_loss = self.loss_fn(g_predictions, real_labels)

        g_optimizer.zero_grad()
        self.manual_backward(g_loss)
        g_optimizer.step()

        # Metrics (a logit > 0 means the discriminator says "real"),
        # averaged over the epoch and shown in the progress bar
        d_real_acc = (real_predictions > 0).float().mean()
        d_fake_acc = (fake_predictions < 0).float().mean()
        self.log_dict(
            {
                "d_loss": d_loss,
                "d_real_acc": d_real_acc,
                "d_fake_acc": d_fake_acc,
                "d_acc": (d_real_acc + d_fake_acc) / 2,
                "g_loss": g_loss,
                "g_acc": (g_predictions > 0).float().mean(),
            },
            prog_bar=True,
            on_step=False,
            on_epoch=True,
        )

    def configure_optimizers(self):
        # One optimizer per network
        d_optimizer = torch.optim.Adam(
            self.discriminator.parameters(), lr=LEARNING_RATE, betas=(ADAM_BETA_1, ADAM_BETA_2)
        )
        g_optimizer = torch.optim.Adam(
            self.generator.parameters(), lr=LEARNING_RATE, betas=(ADAM_BETA_1, ADAM_BETA_2)
        )
        return d_optimizer, g_optimizer

In [ ]:
# Create a DCGAN
dcgan = DCGAN(
    discriminator=discriminator, generator=generator, latent_dim=Z_DIM
)

In [ ]:
if LOAD_MODEL:
    dcgan = DCGAN.load_from_checkpoint(
        "./checkpoint/checkpoint.ckpt", discriminator=discriminator, generator=generator
    )

## 3. Train the GAN <a name="train"></a>

In [ ]:
# The optimizers are defined in DCGAN.configure_optimizers() above;
# the Trainer calls it when training starts. Let's have a look at them:
d_optimizer, g_optimizer = dcgan.configure_optimizers()
print(d_optimizer)
print(g_optimizer)

In [ ]:
# Create a model save checkpoint (overwritten at the end of every epoch)
model_checkpoint_callback = ModelCheckpoint(
    dirpath="./checkpoint",
    filename="checkpoint",
    every_n_epochs=1,
    enable_version_counter=False,
)

tensorboard_logger = TensorBoardLogger(save_dir="./logs")


class ImageGenerator(L.Callback):
    def __init__(self, num_img, latent_dim):
        self.num_img = num_img
        self.latent_dim = latent_dim

    def on_train_epoch_end(self, trainer, pl_module):
        generator = pl_module.generator
        generator.eval()  # BatchNorm uses its running statistics, like Keras' training=False
        with torch.no_grad():
            random_latent_vectors = torch.randn(self.num_img, self.latent_dim, device=pl_module.device)
            generated_images = generator(random_latent_vectors)
        generator.train()
        generated_images = generated_images * 127.5 + 127.5
        display(
            generated_images,
            save_to="./output/generated_img_%03d.png" % (trainer.current_epoch),
        )

In [ ]:
trainer = L.Trainer(
    max_epochs=EPOCHS,
    accelerator="auto",
    logger=tensorboard_logger,
    callbacks=[
        model_checkpoint_callback,
        ImageGenerator(num_img=10, latent_dim=Z_DIM),
        # The plain tqdm progress bar: the Rich one (used when `rich` is installed)
        # clashes with printing/plotting from a callback inside Jupyter
        TQDMProgressBar(),
    ],
)
trainer.fit(dcgan, train)

In [ ]:
# Save the final models
torch.save(generator.state_dict(), "./models/generator.pt")
torch.save(discriminator.state_dict(), "./models/discriminator.pt")

## 3. Generate new images <a name="decode"></a>

In [ ]:
# Sample some points in the latent space, from the standard normal distribution
grid_width, grid_height = (10, 3)
z_sample = np.random.normal(size=(grid_width * grid_height, Z_DIM))

In [ ]:
# Decode the sampled points
generator.eval()  # BatchNorm in inference mode, like Keras' predict()
with torch.no_grad():
    z = torch.tensor(z_sample, dtype=torch.float32, device=dcgan.device)
    reconstructions = generator(z).cpu().numpy()  # (N, 1, H, W)

In [ ]:
# Draw a plot of decoded images
fig = plt.figure(figsize=(18, 5))
fig.subplots_adjust(hspace=0.4, wspace=0.4)

# Output the grid of faces
for i in range(grid_width * grid_height):
    ax = fig.add_subplot(grid_height, grid_width, i + 1)
    ax.axis("off")
    ax.imshow(reconstructions[i, 0], cmap="Greys")  # channel 0 of the grayscale image

In [ ]:
def compare_images(img1, img2):
    return np.mean(np.abs(img1 - img2))

In [ ]:
all_data = []
for i in train:
    all_data.extend(i.numpy())
all_data = np.array(all_data)

In [ ]:
r, c = 3, 5
fig, axs = plt.subplots(r, c, figsize=(10, 6))
fig.suptitle("Generated images", fontsize=20)

noise = np.random.normal(size=(r * c, Z_DIM))
generator.eval()
with torch.no_grad():
    gen_imgs = generator(torch.tensor(noise, dtype=torch.float32, device=dcgan.device)).cpu().numpy()

cnt = 0
for i in range(r):
    for j in range(c):
        axs[i, j].imshow(gen_imgs[cnt, 0], cmap="gray_r")
        axs[i, j].axis("off")
        cnt += 1

plt.show()

In [ ]:
fig, axs = plt.subplots(r, c, figsize=(10, 6))
fig.suptitle("Closest images in the training set", fontsize=20)

cnt = 0
for i in range(r):
    for j in range(c):
        c_diff = 99999
        c_img = None
        for k_idx, k in enumerate(all_data):
            diff = compare_images(gen_imgs[cnt], k)
            if diff < c_diff:
                c_img = np.copy(k)
                c_diff = diff
        axs[i, j].imshow(c_img[0], cmap="gray_r")
        axs[i, j].axis("off")
        cnt += 1

plt.show()